# Modèles de machine learning avec Python et SAS Viya

## Bibliothèques

In [ ]:
pip install swat seaborn

In [ ]:
import swat
import numpy as np
from numpy import trapezoid
import pandas as pd
import seaborn as sns
%matplotlib inline
from matplotlib import pyplot as plt
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


## Créer une session CAS

La session CAS ouvre l'accès au serveur et aux bibliothèques autorisées.
C'est le point de départ du flux de préparation et de modélisation.

In [ ]:
session = swat.CAS(
    "https://server.demo.sas.com/cas-shared-default-http",
    authinfo='/Users/sbxxab/.authinfo',
    ssl_ca_list="/Users/sbxxab/Downloads/server.demo.sas.com.pem",
)

## Charger des données externes dans CAS

CAS lit des données depuis plusieurs sources.
Ici, nous chargeons le CSV Titanic depuis GitHub, puis *tbl* référence la table en mémoire.

In [ ]:
data_source = "https://raw.githubusercontent.com/datasciencedojo/datasets/refs/heads/master/titanic.csv"
table_name = "titanic"

tbl = session.read_csv(data_source, table_name)

## Explorer les données

*shape* donne le nombre de lignes et de colonnes.

In [ ]:
tbl.shape

*columninfo* décrit les variables disponibles.

In [ ]:
tbl.columninfo()

*head* affiche un extrait des premières lignes.

In [ ]:
tbl.head(3)

*describe* fournit un résumé statistique rapide.

In [ ]:
tbl.describe(include='all')

## Passer en DataFrame

La conversion de *tbl* en DataFrame facilite l'analyse locale dans le notebook.

In [ ]:
df = tbl.to_frame()

Le DataFrame permet d'enchaîner naturellement avec Pandas et les autres bibliothèques Python.

In [ ]:
distribution_plot  = df.drop('Survived', axis=1).hist(bins=15, figsize=(12,12), alpha=0.75)

Exemple: visualisation de la distribution avec Seaborn.

In [ ]:
# Pour les grilles à facettes Seaborn, créez un graphique vide 3 par 2 sur lequel placer les données
pclass_survived = sns.FacetGrid(
    df, col="Survived", row="Pclass", height=2.5, aspect=3
)

# Superposer un histogramme de Y (Age) = Survived
pclass_survived.map(plt.hist, "Age", alpha=0.75, bins=25)

# Ajouter une légende pour plus de lisibilité
pclass_survived.add_legend()

In [ ]:
# Créer un canevas graphique - Un pour le taux de survie des femmes - Un pour les hommes
Survived = "Survived"
Not_Survived = "Not Survived"
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))

# Initialiser les variables femmes et hommes à la valeur de l'ensemble de données
Women = df[df["Sex"] == "female"]
Male = df[df["Sex"] == "male"]

# Pour le premier graphique, tracez le nombre de femmes qui ont survécu en fonction de leur âge
Female_vs_Male = sns.histplot(
    Women[Women["Survived"] == 1].Age.dropna(),
    bins=25,
    label=Survived,
    ax=axes[0],
    kde=False,
)

# Pour le premier graphique, répartissez le nombre de femmes qui n'ont pas survécu en fonction de leur âge.
Female_vs_Male = sns.histplot(
    Women[Women["Survived"] == 0].Age.dropna(),
    bins=25,
    label=Not_Survived,
    ax=axes[0],
    kde=False,
)

# Afficher une légende pour le premier graphique
Female_vs_Male.legend()
Female_vs_Male.set_title("Females")

# Pour le deuxième graphique, tracez le nombre d'hommes qui ont survécu en fonction de leur âge
Female_vs_Male = sns.histplot(
    Male[Male["Survived"] == 1].Age.dropna(),
    bins=25,
    label=Survived,
    ax=axes[1],
    kde=False,
)

# Pour le deuxième graphique, répartissez le nombre d'hommes qui n'ont pas survécu en fonction de leur âge.
Female_vs_Male = sns.histplot(
    Male[Male["Survived"] == 0].Age.dropna(),
    bins=25,
    label=Not_Survived,
    ax=axes[1],
    kde=False,
)

# Afficher une légende pour le deuxième graphique
Female_vs_Male.legend()
Female_vs_Male.set_title("Men")

In [ ]:
# Matrice de corrélation Heatmap qui compare les valeurs numériques et "a survécu"
Heatmap_Matrix = sns.heatmap(
    df[["Survived", "SibSp", "Parch", "Age", "Fare"]].corr(),
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0,
    linewidths=0.1,
)

## Traiter les valeurs manquantes

Avant l'entraînement, on nettoie les données.
Ici, on applique l'imputation CAS avec la configuration par défaut.

In [ ]:
tbl.distinct()

Pour ce jeu de données, l'action *impute* suffit pour compléter les valeurs manquantes.

In [ ]:
session.dataPreprocess.impute(
    table = 'titanic',
    inputs = ['Age'],
    copyAllVars = True,
    casOut = 'titanic_imputed',
)

## Recharger localement pour créer des variables

On récupère la table imputée, puis on la convertit en DataFrame.

In [ ]:
tbl_i = session.CASTable('titanic_imputed')
df_i = tbl_i.to_frame()

Supprimer les variables non nécessaires à la prédiction

In [ ]:
df_i = df_i[
    ["Embarked", "Parch", "Sex", "Pclass", "SibSp", "Survived", "Fare", "IMP_Age"]
]

Nous vérifions ensuite que l'imputation CAS a bien été appliquée.

Compter les valeurs nulles avec *isnull()*

In [ ]:
total_missing = df_i.isnull().sum().sort_values(ascending=False)
total_missing.head(5)

Calculer le total des valeurs présentes

In [ ]:
total = df_i.notnull().sum().sort_values(ascending=False)
total.head(5)

Calculer le pourcentage de valeurs manquantes par variable

In [ ]:
Percent = df_i.isnull().sum() / df_i.isnull().count() * 100
Percent.sort_values(ascending=False).head(5)

Arrondir les pourcentages pour une lecture plus claire

In [ ]:
Percent_Rounded = (round(Percent, 1)).sort_values(ascending=False)

Afficher une vue combinée: total, manquants et pourcentage

In [ ]:
Missing_Data = pd.concat(
    [total, total_missing, Percent_Rounded],
    axis=1,
    keys=["Non Missing Values", "Total Missing Values", "% Missing"],
    sort=True,
)
Missing_Data

## Créer de nouvelles variables

L'ingénierie de variables améliore souvent la performance.
Dans cette démo, nous créons:

- *Relatives*
- *Alone_on_Ship*
- *Age_Times_Class*
- *Fare_Per_Person*

Objectif: mieux capturer la situation de chaque passager.

In [ ]:
data = [df_i]
for dataset in data:
    dataset['Relatives'] = dataset['SibSp'] + dataset['Parch']
    dataset.loc[dataset['Relatives'] > 0, 'Alone_On_Ship'] = 0
    dataset.loc[dataset['Relatives'] == 0, 'Alone_On_Ship'] = 1
    dataset['Alone_On_Ship'] = dataset['Alone_On_Ship'].astype(int)

    dataset["Age_Times_Class"] = dataset["IMP_Age"] * dataset["Pclass"]

    dataset["Fare_Per_Person"] = dataset["Fare"] / (dataset["Relatives"] + 1)
    dataset["Sib_Div_Spouse"] = dataset["SibSp"]
    dataset["Parents_Div_Children"] = dataset["Parch"]

# Supprimez les fonctionnalités inutiles
df_i = df_i.drop(["SibSp", "Parch"], axis=1)
# Afficher quelques données
df_i.head(5)

## Recharger les données dans CAS pour l'entraînement

In [ ]:
session.upload_frame(df_i, casout="titanic_prepared")

## Créer les jeux train/test

La partition est automatisée avec *sampling.srs*.
L'indicateur *_PartInd_* vaut 1 pour train et 0 pour test.

In [ ]:
session.loadactionset("sampling")

session.sampling.srs(table="titanic_prepared", samppct=80, partind= True, output = dict(casout="titanic_train", copyVars = 'All'))

## Entraîner plusieurs modèles CAS

Nous comparons plusieurs approches (Forest, Decistion Tree, Gradient Boosting).
Étapes: charger les actions, définir cible/entrées/nominales, entraîner.

In [ ]:
# Charger l'ensemble d'actions CAS de DecisionTree
session.loadActionSet("decisionTree")

# Définir un objectif pour la modélisation prédictive
target = "Survived"

# Définir les entrées à utiliser pour prédire la survie (entrées de variables numériques)
inputs = [
    "Sex",
    "Pclass",
    "Alone_On_Ship",
    "Age_Times_Class",
    "Relatives",
    "IMP_Age",
    "Fare",
    "Fare_Per_Person",
    "Embarked",
]

# Définir les variables nominales à utiliser dans le modèle (entrées de variables catégories)
nominals = ["Sex", "Pclass", "Alone_On_Ship", "Embarked", "Survived"]

# Entraîner le modèle Forest
session.decisionTree.forestTrain(
    table=dict(name="titanic_train", where="_PartInd_ = 1"),
    target=target,
    inputs=inputs,
    nominals=nominals,
    casOut=dict(name="titanic_forest_model", replace=True),
)

Pourquoi une clause WHERE sur la table d'entrée?
Parce que l'entraînement doit utiliser uniquement les lignes marquées train par *_PartInd_*.
La sortie CAS donne aussi des métriques utiles pour le réglage.

In [ ]:
# Entraîner le modèle Decision Tree
session.decisionTree.dtreeTrain(
    table=dict(name="titanic_train", where="_PartInd_ = 1"),
    target=target,
    inputs=inputs,
    nominals=nominals,
    casOut=dict(name="titanic_decisiontree_model", replace=True),
)

In [ ]:
# Entraîner le modèle Gradient Boosting Tree
session.decisionTree.gbtreeTrain(
    table=dict(name="titanic_train", where="_PartInd_ = 1"),
    target=target,
    inputs=inputs,
    nominals=nominals,
    casOut=dict(name="titanic_gradient_model", replace=True),
)

## Noter les modèles

Chaque action de scoring produit une table de prédictions et d'indicateurs.

In [ ]:
titanic_forest_score = session.decisionTree.forestScore(
    table=dict(name="titanic_train", where="_PartInd_ = 0"),
    model="titanic_forest_model",
    casout=dict(name="titanic_forest_score", replace=True),
    copyVars=target,
    encodename=True,
    assessonerow=True,
)

titanic_decisiontree_score = session.decisionTree.dtreeScore(
    table=dict(name="titanic_train", where="_PartInd_ = 0"),
    model="titanic_decisiontree_model",
    casout=dict(name="titanic_decisiontree_score", replace=True),
    copyVars=target,
    encodename=True,
    assessonerow=True,
)

titanic_gradient_score = session.decisionTree.gbtreeScore(
    table=dict(name="titanic_train", where="_PartInd_ = 0"),
    model="titanic_gradient_model",
    casout=dict(name="titanic_gradient_score", replace=True),
    copyVars=target,
    encodename=True,
    assessonerow=True,
)

Le scoring crée notamment *P_survived1* et *P_survived0*.
Ces probabilités servent à évaluer la qualité du classement sur le jeu test.

In [ ]:
titanic_forest_score

À partir de l'erreur de classification, on estime une précision globale avec $1 - \text{erreur}$.
C'est utile, mais à compléter par l'analyse des faux positifs/faux négatifs.

In [ ]:
session.loadActionSet("percentile")
prediction = "P_survived1"

titanic_forest_assessed = session.percentile.assess(
    table="titanic_forest_score",
    inputs=prediction,
    casout=dict(name="titanic_forest_assessed", replace=True),
    includeRoc=True,
    response=target,
    event="1",
)

titanic_decisiontree_assessed = session.percentile.assess(
    table="titanic_decisiontree_score",
    inputs=prediction,
    casout=dict(name="titanic_decisiontree_assessed", replace=True),
    includeRoc=True,
    response=target,
    event="1",
)

titanic_gradient_assessed = session.percentile.assess(
    table="titanic_gradient_score",
    inputs=prediction,
    casout=dict(name="titanic_gradient_assessed", replace=True),
    includeRoc=True,
    response=target,
    event="1",
)

L'évaluation CAS retourne trois familles d'indicateurs:

- lift;
- ROC;
- concordance.

In [ ]:
titanic_decisiontree_assessed

## Analyser les résultats localement

On trace les courbes ROC et de lift pour comparer les modèles, puis on calcule l'AUC pour une synthèse globale.

In [ ]:
# Évaluer le modèle Forest
titanic_assess_ROC_Forest = session.CASTable("titanic_forest_assessed_ROC")
titanic_assess_Lift_Forest = session.CASTable("titanic_forest_assessed")

titanic_ROC_pandas_Forest = titanic_assess_ROC_Forest.to_frame()
titanic_Lift_pandas_Forest = titanic_assess_Lift_Forest.to_frame()

# Évaluer le modèle Decision Tree
titanic_assess_ROC_DT = session.CASTable("titanic_decisiontree_assessed_ROC")
titanic_assess_Lift_DT = session.CASTable("titanic_decisiontree_assessed")

titanic_ROC_pandas_DT = titanic_assess_ROC_DT.to_frame()
titanic_Lift_pandas_DT = titanic_assess_Lift_DT.to_frame()

# Évaluer le modèle Gradient Boosting Tree
titanic_assess_ROC_gb = session.CASTable("titanic_gradient_assessed_ROC")
titanic_assess_Lift_gb = session.CASTable("titanic_gradient_assessed")

titanic_ROC_pandas_gb = titanic_assess_ROC_gb.to_frame()
titanic_Lift_pandas_gb = titanic_assess_Lift_gb.to_frame()

Une fois les tables connectées, on trace toutes les courbes ROC sur un même graphique.

In [ ]:
# Tracer le ROC localement
plt.figure(figsize=(10, 10))
plt.plot(
    1 - titanic_ROC_pandas_Forest["_Specificity_"],
    titanic_ROC_pandas_Forest["_Sensitivity_"],
    "bo-",
    linewidth=3,
)
plt.plot(
    1 - titanic_ROC_pandas_DT["_Specificity_"],
    titanic_ROC_pandas_DT["_Sensitivity_"],
    "ro-",
    linewidth=3,
)
plt.plot(
    1 - titanic_ROC_pandas_gb["_Specificity_"],
    titanic_ROC_pandas_gb["_Sensitivity_"],
    "go-",
    linewidth=3,
)
plt.plot(pd.Series(range(0, 11, 1)) / 10, pd.Series(range(0, 11, 1)) / 10, "k--")
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.legend(["Forest", "Decision Tree", "Gradient Boosting"])
plt.show()

On peut aussi comparer les courbes de lift cumulées.

In [ ]:
# Lift
plt.figure(figsize=(10, 10))
plt.plot(
    titanic_Lift_pandas_Forest["_Depth_"],
    titanic_Lift_pandas_Forest["_CumLift_"],
    "bo-",
    linewidth=3,
)
plt.plot(
    titanic_Lift_pandas_DT["_Depth_"],
    titanic_Lift_pandas_DT["_CumLift_"],
    "ro-",
    linewidth=3,
)
plt.plot(
    titanic_Lift_pandas_gb["_Depth_"],
    titanic_Lift_pandas_gb["_CumLift_"],
    "go-",
    linewidth=3,
)
plt.xlabel("Depth")
plt.ylabel("Cumulative Lift")
plt.title("Cumulative Lift Curve")
plt.legend(["Forest", "Decision Tree", "Gradient Boosting"])
plt.show()

Pour une présentation, l'AUC est l'indicateur le plus lisible: plus elle est élevée, meilleure est la séparation des classes.

In [ ]:
# Scores du modèle Forest
x_forest = np.array([titanic_ROC_pandas_Forest["_Specificity_"]])
y_forest = np.array([titanic_ROC_pandas_Forest["_Sensitivity_"]])

# Scores du modèle Decision Tree
x_dt = np.array([titanic_ROC_pandas_DT["_Specificity_"]])
y_dt = np.array([titanic_ROC_pandas_DT["_Sensitivity_"]])

# Scores du modèle Gradient Boosting
x_gb = np.array([titanic_ROC_pandas_gb["_Specificity_"]])
y_gb = np.array([titanic_ROC_pandas_gb["_Sensitivity_"]])

# Calculer l'aire sous la courbe (intégrer)
area_forest = trapezoid(y_forest, x_forest)
area_dt = trapezoid(y_dt, x_dt)
area_gb = trapezoid(y_gb, x_gb)

# Tableau des scores du modèle
Model_Results = pd.DataFrame(
    {
        "Model": ["Forest", "Decision Tree", "Gradient Boosting"],
        "Score": [area_forest, area_dt, area_gb],
    }
)

Model_Results

Le score AUC ROC résume la capacité du modèle à distinguer survivants et non-survivants.

In [ ]:
session.close()